# 08 · Own pipeline, part 3: a stronger RT-DETR

Notebook 07 showed that the pipeline is limited by its strongest member: flip TTA, the ensemble and
temporal rescoring together add +0.017 val mAP to RT-DETR (0.256 → 0.273). This notebook improves the
model itself, one change per run, so the effect of each change is measured on its own:

| run | change | motivated by |
|---|---|---|
| `rtdetr_data3` | every 3rd training frame (8,090) instead of every 10th (2,446), 4 epochs | RT-DETR peaked after 1–2 of 8 epochs in notebook 05: it saw the same frames too often |
| `rtdetr_hires` | + input 1080 × 1920 instead of 800 × 1333, 2 epochs | small objects are the weakest point (AP_small 0.145 vs AP_large 0.44) |
| `rtdetr_lowres` | + 30% of training frames shrunk to 680 or 960 px wide | 16% of the test frames have these widths; train and val have none below 1344 px |
| `rtdetr_r101` | ResNet-101 backbone instead of ResNet-50, with the better of the two settings above | a deeper, different model for the ensemble; Faster R-CNN and FCOS made it worse in notebook 07 |
| `rtdetr_v2` | RT-DETRv2 (Lv et al., 2024, Baidu), same setting | a third strong model, with an improved decoder and training recipe |

**Val is not like test.** All 3,811 test frames are at most 1920 px wide (16% of them only 680–960 px),
while 55% of the val frames are 2.7K–4K. A change can help on test but not on val: `rtdetr_data3` gained
+0.005 val mAP but +0.022 on the public leaderboard. Next to the usual full-val metrics, every run therefore
gets a **test-like score**: mAP on the val frames at most 1920 px wide, and on the same frames shrunk to
960 and 680 px wide (weighted 84% / 16%, the test shares). Decisions use this score; the public
leaderboard (30% of the test set) is only a secondary check, as the final grade uses the other 70%.

The runs take longer than one Kaggle session allows, so the notebook runs in several sessions: each
session has a time budget, training stops at an epoch boundary before the budget ends, and the next
session (with the previous output attached as input) continues from the last epoch.

## Setup

In [ ]:
# Setup: clone/update the repo on Colab/Kaggle, install requirements, resolve paths, get the data
import os, sys, subprocess
REPO_URL = "https://github.com/anna-kryvulia/ucu-ai-hw-object-detection.git"
BRANCH = "main"   # branch with the code you pushed
if "google.colab" in sys.modules or os.path.exists("/kaggle/working"):
    ROOT = "/content/repo" if "google.colab" in sys.modules else "/kaggle/working/repo"
    if not os.path.exists(ROOT):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{ROOT}/requirements.txt"], check=True)
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, f"{ROOT}/src")

from hwdet import env
from hwdet.data.download import ensure_data
from hwdet.tracker import Run, status
P = env.setup(seed=42)
ensure_data(P, drive_cache=False)   # Colab: the zip goes to the local disk, not Drive
status(P)                           # which runs / stages are already done

In [ ]:
# Imports, plot style, W&B login, time budget of this session
import json
import time
import pandas as pd
import matplotlib.pyplot as plt
import torch

from hwdet.config import load_config
from hwdet.data.detection_dataset import DetectionDataset, build_targets
from hwdet.data.io import frame_table
from hwdet.engine import Data, load_weights, predict, run_experiment
from hwdet.env import REPO_ROOT
from hwdet.eval.coco_eval import evaluate
from hwdet.pipeline import DownscaledDetector
from hwdet.rtdetr import build_rtdetr
from hwdet.viz import save_fig, set_style
from hwdet.wandb_utils import login_from_secrets

SESSION_HOURS = 11.5                     # GPU hours this session may use (Kaggle quota left minus a margin)
NOTEBOOK_START = time.time()
STOP_AT = NOTEBOOK_START + SESSION_HOURS * 3600
set_style()
login_from_secrets()
dev = env.device()
print("torch", torch.__version__, "| device", dev, f"| time budget {SESSION_HOURS} h")

## 1. Data

In [ ]:
# Every 3rd training frame (new) next to every 10th (notebooks 02-05); full val, every 5th val frame to monitor
cfg = load_config("rtdetr_data3")
all_train = frame_table(P.data, "train")
train10 = all_train[(all_train.frame - 1) % 10 == 0]
train3 = all_train[(all_train.frame - 1) % cfg["split"]["train_frame_step"] == 0]
val_frames = frame_table(P.data, "val")
monitor_frames = val_frames[(val_frames.frame - 1) % cfg["split"]["monitor_frame_step"] == 0]
test_frames = frame_table(P.data, "test")
targets = build_targets(P.data, pd.concat([train3, train10, val_frames]).drop_duplicates("file_name"))
data = Data(P.data, train3, val_frames, monitor_frames, targets, test_frames)
n_obj = {fn: len(t["boxes"]) for fn, t in targets.items()}
pd.DataFrame({"frames": [len(train10), len(train3), len(val_frames), len(test_frames)],
              "objects": [sum(n_obj.get(f, 0) for f in d.file_name) for d in (train10, train3, val_frames)] + [None]},
             index=["train, every 10th", "train, every 3rd", "val", "test"])

In [ ]:
# Frames per width: test has low resolutions (680, 960 px) that train and val do not have, val has 4K that test does not
w = pd.concat([train3.width, val_frames.width, test_frames.width], keys=["train, every 3rd", "val", "test"])
group = w.map(lambda x: "680-960" if x < 1000 else "1344-1360" if x < 1400 else "1904-1920" if x < 1950 else "2688-3840")
pd.crosstab(w.index.get_level_values(0), group).loc[["train, every 3rd", "val", "test"]]

## 2. Test-like score

Val frames at most 1920 px wide (1,286 frames, the 1344 and 1904 px clips) are scored as they are, and
every 2nd of them is also shrunk to 960 and 680 px wide before the model (boxes scaled back), as a stand-in
for the low-resolution test clips. Test-like mAP = 0.84 · mAP(≤ 1920 px) + 0.16 · mean(mAP at 960, mAP at 680).
The score of each finished run is saved next to it (`test_like.json`), so it is computed once.

In [ ]:
# Test-like score of a finished run (needs its checkpoint and val predictions)
like_test = val_frames[val_frames.width <= 1920]
probe_frames = like_test.iloc[::2]
LOW_SHARE = 0.16                          # share of test frames 680-960 px wide


def score(preds, frames):
    return evaluate(preds[preds.file_name.isin(frames.file_name)], frames, P.data, per_class=False)["mAP"]


def test_like(run_id):
    f = P.runs / run_id / "test_like.json"
    if f.exists():
        return json.loads(f.read_text())
    out = {"mAP ≤ 1920 px": score(pd.read_csv(P.runs / run_id / "predictions" / "val.csv"), like_test)}
    model = load_weights(build_rtdetr(load_config(run_id)), Run(run_id, None, P), "best", dev)
    ds = DetectionDataset(probe_frames, P.data, targets)
    for w in (960, 680):
        p = predict(DownscaledDetector(model, w), ds, dev, batch_size=4, workers=2, desc=f"{run_id} at {w} px")
        out[f"mAP at {w} px"] = score(p, probe_frames)
    out["test-like mAP"] = (1 - LOW_SHARE) * out["mAP ≤ 1920 px"] + LOW_SHARE * (out["mAP at 960 px"] + out["mAP at 680 px"]) / 2
    f.write_text(json.dumps(out, indent=2))
    del model
    torch.cuda.empty_cache()
    return out


print(len(like_test), "val frames at most 1920 px wide,", len(probe_frames), "of them shrunk for the low-resolution probe")

## 3. Model check

Every architecture used below is built once on the CPU before any training starts, so a wrong checkpoint
name or an incompatible library version shows up in the first minutes instead of after hours of training.

In [ ]:
# Build each model once (COCO weights + new 8-class heads) and count its parameters
for name in ("rtdetr_data3", "rtdetr_r101", "rtdetr_v2"):
    c = load_config(name)
    m = build_rtdetr(c)
    print(f"{name}: {c['model']['arch']}, {sum(p.numel() for p in m.parameters()) / 1e6:.1f} M parameters")
    del m

## 4. Runs

Each cell trains one config, evaluates the best epoch on the full val split, predicts the test set and
writes a submission file. A run starts only after the previous one has finished; a finished run only
prints its metrics, and a run reached after the time budget prints that it was postponed.

In [ ]:
# Run 1: every 3rd training frame (3.4x more distinct frames), 4 epochs, input 800 x 1333 as in notebook 05
runs = {}
runs["rtdetr_data3"] = run_experiment("rtdetr_data3", build_rtdetr, P, data, predict_test=True,
                                      stop_at=STOP_AT, eval_hours=0.6)

In [ ]:
# Run 2: as run 1 with input 1080 x 1920 (test frames at full size), batch size 2, 2 epochs
if runs["rtdetr_data3"].is_done("test"):
    runs["rtdetr_hires"] = run_experiment("rtdetr_hires", build_rtdetr, P, data, predict_test=True,
                                          stop_at=STOP_AT, eval_hours=1.0)
else:
    print("rtdetr_hires waits for rtdetr_data3")

In [ ]:
# Run 3: as run 2 with 30% of the training frames shrunk to 680 or 960 px wide (the low test resolutions)
if "rtdetr_hires" in runs and runs["rtdetr_hires"].is_done("test"):
    runs["rtdetr_lowres"] = run_experiment("rtdetr_lowres", build_rtdetr, P, data, predict_test=True,
                                           stop_at=STOP_AT, eval_hours=1.0)
else:
    print("rtdetr_lowres waits for rtdetr_hires")

In [ ]:
# Low-resolution augmentation for the next runs only if run 3 beat run 2 on the test-like score
if "rtdetr_lowres" in runs and runs["rtdetr_lowres"].is_done("test"):
    hires, lowres = test_like("rtdetr_hires"), test_like("rtdetr_lowres")
    use_lowres = lowres["test-like mAP"] > hires["test-like mAP"]
    print(f"test-like mAP: rtdetr_hires {hires['test-like mAP']:.4f}, rtdetr_lowres {lowres['test-like mAP']:.4f}"
          f" -> next runs {'with' if use_lowres else 'without'} the low-resolution augmentation")


def next_run(name):
    """Overrides for runs 4-5: the low-resolution augmentation if it helped."""
    if not use_lowres:
        return {}
    return {"train": {"lowres": 0.3, "lowres_widths": [680, 960]},
            "description": load_config(name)["description"] + ", 30% of frames shrunk to 680 or 960 px"}

In [ ]:
# Run 4: RT-DETR with a ResNet-101 backbone at 1080 x 1920
if "rtdetr_lowres" in runs and runs["rtdetr_lowres"].is_done("test"):
    runs["rtdetr_r101"] = run_experiment("rtdetr_r101", build_rtdetr, P, data, predict_test=True,
                                         stop_at=STOP_AT, eval_hours=1.2, **next_run("rtdetr_r101"))
else:
    print("rtdetr_r101 waits for rtdetr_lowres")

In [ ]:
# Run 5: RT-DETRv2 (ResNet-50) at 1080 x 1920
if "rtdetr_r101" in runs and runs["rtdetr_r101"].is_done("test"):
    runs["rtdetr_v2"] = run_experiment("rtdetr_v2", build_rtdetr, P, data, predict_test=True,
                                       stop_at=STOP_AT, eval_hours=1.0, **next_run("rtdetr_v2"))
else:
    print("rtdetr_v2 waits for rtdetr_r101")

## 5. Results

In [ ]:
# Full val split and test-like score: notebook 05's RT-DETR next to the finished runs of this notebook
cols = ["mAP", "AP50", "AP_small", "AP_medium", "AP_large", "AR_500", "ms_per_img"]
old = pd.read_csv(REPO_ROOT / "results" / "experiments.csv")
rows = {"rtdetr_baseline": old[(old.run_id == "rtdetr_baseline") & (old["eval"] == "val")].iloc[0][cols]}
for r, run in runs.items():
    if run.is_done("test"):
        rows[r] = pd.concat([pd.Series(run.metrics["val"])[cols], pd.Series(test_like(r))])
res = pd.DataFrame(rows).T.rename(columns={"mAP": "val mAP"})
res.insert(1, "val change vs baseline", res["val mAP"] - res.loc["rtdetr_baseline", "val mAP"])
res.round(4)

In [ ]:
# Val mAP on the monitor subset after each epoch, by images seen (epochs differ in length between the runs)
hists = {"rtdetr_baseline": pd.read_csv(REPO_ROOT / "results" / "runs" / "rtdetr_baseline" / "history.csv")}
hists.update({r: run.history for r, run in runs.items() if len(run.history)})
n_train = {"rtdetr_baseline": len(train10)}
fig, ax = plt.subplots(figsize=(7, 4))
for r, h in hists.items():
    seen = (h.epoch + 1) * n_train.get(r, len(train3)) / 1000
    ax.plot(seen, h.val_mAP, marker="o", label=r)
ax.set_xlabel("training images seen (thousands)")
ax.set_ylabel("val mAP (monitor subset)")
ax.legend()
ax.grid(alpha=0.3)
save_fig(fig, "rtdetr_scaled_curves", P)
plt.show()

In [ ]:
# Status of every run, time used, distinct warnings logged in this session (should be empty)
print(f"session: {(time.time() - NOTEBOOK_START) / 3600:.2f} h of {SESSION_HOURS} h")
print({r: list(run.stages) for r, run in runs.items()})
env.warnings_summary(P)